In [ ]:
import pandas as pd
import sklearn as sk
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

#Others's non-exploratory code to reuse
df_raw = pd.read_csv('EPA_SmartLocationDatabase_V3_Jan_2021_Final.csv')
df_clean = df_raw[df_raw['TotPop'] > 0]
df_clean = df_clean[df_clean['TotEmp'] > 0]

cols_to_keep = [
    # Geographic Identifiers
    'GEOID10', # Census block group 12-digit FIPS code (2010)
    'GEOID20', # Census block group 12-digit FIPS code (2018)
    'CBSA', # FIPS for Core-Based Statistical Area (CBSA) in which CBG resides

    # Candidate Response Variable(s)
    'D5AR', # Jobs within 45 minutes auto travel time,
    'D5BR', # Jobs within 45-minute transit commute,
    
    # Transit Predictors
    'D4A', # Distance from the population-weighted centroid to nearest transit stop (meters)
    'D4C', # Aggregate frequency of transit service within 0.25 miles of CBG boundary per hour during evening peak period
    'D4D', # Aggregate frequency of transit service [D4c] per square mile
    'D4E', # Aggregate frequency of transit service [D4c] per capita
    
    # Other Variables/Controls
    
    ## Demographics
    'TotPop', # Population, 2018
    'HH', # Households (occupied housing units), 2018
    'P_WrkAge', # Percent of population that is working aged 18 to 64 years,
    'Pct_AO0',  # zero-car households (equity weight)
    'Workers', # Count of workers in CBG (home location), 2017
    'R_PCTLOWWAGE', # Percent of low wage workers in a CBG (home location), 2017
    
    ## Employment
    'TotEmp', # Total employment, 2017
    
    ## Density (D1)
    'D1B', # Gross population density (people/acre)
    'D1C', # Gross employment density (jobs/acre)
    
    ## Design (D3)
    'D3A', # Total road network density
    'D3B', # Street intersection density (weighted, auto-oriented intersections eliminated)
]

df_select = df_clean.copy()
df_select = df_select[cols_to_keep]

df_clean = df_select.copy()
df_clean = df_clean[df_clean['D5BR'] != -99999].copy()

# For 'D4A', set rows with no nearby transit to NA and add indicator
df_clean["D4A_no_transit"] = (df_clean["D4A"] == -99999).astype(int)
df_clean["D4A"] = df_clean["D4A"].replace(-99999, np.nan)

# For 'D4C', 'D4D', and 'D4E', set sentinel values to 0 and add indicator
for col in ["D4C", "D4D", "D4E"]:
    df_clean[col + "_no_transit"] = (df_clean[col] == -99999).astype(int)
    df_clean[col] = df_clean[col].replace(-99999, 0)

log_cols = ["D5AR", "D5BR", "D4A", "TotEmp", "Workers", "TotPop", "HH","D1C", "D1B", "D3B"]

for col in log_cols:
    df_clean["log_" + col] = np.log1p(df_clean[col])


df_clean.head()

Utilizing the code used previously to generate datasets to be used.
Delete from here up before uploading, look up previously used variable names to keep consistent 

Goal of Regression is to find more values that help predict the where the areas of opportunity are as defined by the created equation. Code section below is to calculate that statistic for each 

R_PCTLOWWAGE * (P_WrkAge - (Workers / Tot_Pop)) * (1-TER ) = Opp_Score 

Demographics * Employment * Transit Accessibility = Opportunity Score 
With a higher score = better areas of equitable opportunity 

In [ ]:

#Create Opportunity Score
pd.set_option('display.max_columns', None)
df_clean["Opp_Score"] = (df_clean["R_PCTLOWWAGE"] * (df_clean["P_WrkAge"] - (df_clean["Workers"] / df_clean["TotPop"])) * (1 - (df_clean["D5BR"] / df_clean["D5AR"])))

df_raw = df_raw[df_raw['TotPop'] > 0]
df_raw = df_raw[df_raw['TotEmp'] > 0]
df_raw = df_raw.drop(['CSA', 'CSA_Name', 'CBSA', 'CBSA_Name'], axis=1) #drop string columns
df_raw["Opp_Score"] = (df_raw["R_PCTLOWWAGE"] * (df_raw["P_WrkAge"] - (df_raw["Workers"] / df_raw["TotPop"])) * (1 - (df_raw["D5BR"] / df_raw["D5AR"])))

df_reduced = df_raw.copy()
df_reduced = df_reduced.drop(["R_PCTLOWWAGE", "P_WrkAge", "Workers", "TotPop", "D5BR", "D5AR"], axis=1)

print(df_raw['Opp_Score'].describe())
sns.boxplot(df_clean, x='Opp_Score')

#shows some pretty extreme outliers on both


Code below this is for creating the different datasets to be used
Raw/All Values are used as a baseline to compare if our selection does improve modeling

In [33]:
#move this to the dataset creation so the calculations can be made 
df_cleaned_X = df_clean[
    [
        "log_D4A", "D4C", "D4D", "D4E", # transit
        "log_TotPop", # population
        "P_WrkAge", "Pct_AO0", # demographics
        "log_D1B", "log_D1C", # density
        "log_D3B", "D3A", # built environment
    ]
]
df_cleaned_y = df_clean["Opp_Score"]
df_reduced_y = df_reduced['Opp_Score']
df_og_y = df_raw['Opp_Score']

df_og_X = df_raw.drop(['Opp_Score'], axis=1)
df_reduced_X = df_reduced.drop(['Opp_Score'], axis=1)


#creating a 2nd with all values except the ones we u

from sklearn.model_selection import train_test_split

X_clean_train, X_clean_test, y_clean_train, y_clean_test = train_test_split(df_cleaned_X, df_cleaned_y, test_size=0.2, random_state=42)

X_clean_train, X_clean_val, y_clean_train, y_clean_val = train_test_split(X_clean_train, y_clean_train, test_size=0.25, random_state=42) # 0.25 x 0.8 = 0.2


X_og_train, X_og_test, y_og_train, y_og_test = train_test_split(df_og_X, df_og_y, test_size=0.2, random_state=42)

X_og_train, X_og_val, y_og_train, y_og_val = train_test_split(X_og_train, y_og_train, test_size=0.25, random_state=42) 


X_reduced_train, X_reduced_test, y_reduced_train, y_reduced_test = train_test_split(df_reduced_X, df_reduced_y, test_size=0.2, random_state=42)



Models Created:

og - All data, none removed or transformed beyond needs to create the metrics - Using DF Clean only ids 
cleaned - Data cleaned for ones with multi-collinearity or deemed unimportant 
reduced - All data except features used in the calculations in order to find alternate features for future/smaller budgets 

LASSO - Cleaned data with LASSO applied 
Elastic Net - Cleaned data with Elastic Net Applied 

In [41]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score


print("Original Dataset MLR Model")
og_model = LinearRegression()
og_model.fit(X_og_train, y_og_train)
target_predicted_og = og_model.predict(X_og_train)

og_model_coefs = pd.DataFrame(zip(X_og_train.columns, og_model.coef_))
print(og_model_coefs)

print(
    "Mean squared error on the training set: "
    f"{mean_squared_error(y_og_train, target_predicted_og):.3f}" #5415.492
)
print("")

print(
    "R2 score on the training set:"
    f"{r2_score(y_og_train,target_predicted_og):.3f}"
)

print("")

print("Clean MLR Model")
clean_model = LinearRegression()
clean_model.fit(X_clean_train, y_clean_train)
target_predicted_clean = clean_model.predict(X_clean_train)
clean_model_coefs = pd.DataFrame(zip(X_clean_train.columns, clean_model.coef_))
print(clean_model_coefs)
print(
    "Mean squared error on the training set: "
    f"{mean_squared_error(y_clean_train, target_predicted_clean):.3f}" #0.099
)
print("")

print(
    "R2 score on the training set:"
    f"{r2_score(y_clean_train,target_predicted_clean):.3f}"
)

print("")

print("Basic Reduced Model")
reduced_model = LinearRegression()
reduced_model.fit(X_reduced_train, y_reduced_train)
target_predicted_reduced = reduced_model.predict(X_reduced_train)
reduced_model_coefs = pd.DataFrame(zip(X_reduced_train.columns, reduced_model.coef_))
print(reduced_model_coefs)
print(
    "Mean squared error on the training set: "
    f"{mean_squared_error(y_reduced_train, target_predicted_reduced):.3f}" #5415.929
)
print("")

print(
    "R2 score on the training set:"
    f"{r2_score(y_reduced_train,target_predicted_reduced):.3f}"
)

print("")


#LASSO alphas to test 
print("LASSO Model")
lasso_model = LassoCV(cv=5, tol=0.001) #tolerance increased due to timeout errors
lasso_model.fit(X_reduced_train, y_reduced_train)
target_predicted_lasso = lasso_model.predict(X_reduced_train)
lasso_model_coefs = pd.DataFrame(zip(X_reduced_train.columns, lasso_model.coef_))
print(lasso_model_coefs)

"""
sfm = SelectFromModel(lasso_model, prefit=True)
X_train_selected = sfm.transform(X_reduced_train.values)
Tested but did not use because it considered the data too noisy or alpha too strict. However, since only GEOID10 and GEOID20 had any coefficients, effectively these are selected. 
"""

print(
    "Mean squared error on the training set: "
    f"{mean_squared_error(y_reduced_train, target_predicted_lasso):.3f}" #5435.713
)
print("")

print(
    "R2 score on the training set:"
    f"{r2_score(y_reduced_train,target_predicted_lasso):.3f}"
)

print("")

#Elastic Net 
print("Elastic Net Model")
elastic_model = ElasticNet(random_state=42)
elastic_model.fit(X_reduced_train, y_reduced_train)
pred_elastic = elastic_model.predict(X_reduced_train)

elastic_model_coefs = pd.DataFrame(zip(X_reduced_train.columns, elastic_model.coef_))
print(elastic_model_coefs)

print(
    "Mean squared error on the training set: "
    f"{mean_squared_error(y_reduced_train, pred_elastic):.3f}" #5435.713
)
print("")

print(
    "R2 score on the training set:"
    f"{r2_score(y_reduced_train,pred_elastic):.3f}"
)

Original Dataset MLR Model
                0             1
0        OBJECTID -1.404601e-06
1         GEOID10  1.149932e-09
2         GEOID20 -1.151575e-09
3         STATEFP  1.450421e-06
4        COUNTYFP -1.450449e-03
5         TRACTCE -6.346085e-07
6        BLKGRPCE  4.073377e-05
7        CBSA_POP -6.911995e-07
8        CBSA_EMP  9.649132e-07
9        CBSA_WRK  4.372158e-07
10       Ac_Total -1.737391e-05
11       Ac_Water -7.075196e-05
12        Ac_Land  5.337818e-05
13        Ac_Unpr  2.219471e-05
14         TotPop  2.054916e-03
15        CountHU  2.276130e-03
16             HH -1.515759e-03
17       P_WrkAge -3.046136e-06
18       AutoOwn0  8.805231e-04
19        Pct_AO0  6.151630e-06
20       AutoOwn1 -1.166911e-04
21        Pct_AO1  5.901247e-06
22      AutoOwn2p -2.279591e-03
23       Pct_AO2p -1.073713e-05
24        Workers -4.024520e-03
25    R_LowWageWk -2.815714e-03
26    R_MedWageWk -2.022666e-03
27     R_HiWageWk  8.138595e-04
28   R_PCTLOWWAGE -6.398047e-07
29         To

/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.590e+08, tolerance: 7.208e+04
  model = cd_fast.enet_coordinate_descent(


Results of Training MSEs 
FIX R^2
	OG	        Clean 	Reduced	    LASSO	    Elastic Net
MSE	5415.492	0.099	4170.786	4182.349	4170.074
R^2	0.004	    0.314	0.004	    0.001	    0.004

Final Comparison Set for which improves the most 


Predict on test, save to y_preds for each model

Calculate MSE, R2

Do ANOVA of across all? May not be accurate due to less features in clean

In [ ]:
#Predictions 
pred_og = og_model.predict(X_og_test)
pred_clean = clean_model.predict(X_clean_test)
pred_reduced = reduced_model.predict(X_reduced_test)
pred_lasso = lasso_model.predict(X_reduced_test)
pred_elastic = elastic_model.predict(X_reduced_test)


#MSE Values
mse_og = mean_squared_error(y_og_test, pred_og)
print(
    "Mean squared error on the OG Test set: "
    f"{mse_og:.3f}" #623.246
)
print("")
mse_clean = mean_squared_error(y_clean_test, pred_clean)
print(
    "Mean squared error on the Clean Test set: "
    f"{mse_clean:.3f}" #0.011
)
print("")
mse_reduced = mean_squared_error(y_reduced_test, pred_reduced)
print(
    "Mean squared error on the Reduced Test set: "
    f"{mse_reduced:.3f}" #622.856
)
print("")
mse_lasso = mean_squared_error(y_reduced_test, pred_lasso)
print(
    "Mean squared error on the LASSO Test set: "
    f"{mse_lasso:.3f}" #627.375
)
print("")
mse_elastic = mean_squared_error(y_reduced_test, pred_elastic)
print(
    "Mean squared error on the Elastic Test set: "
    f"{mse_elastic:.3f}" #622.529
)
print("")

#R^2 Values
r2_og = r2_score(y_og_test, pred_og)
print(
    "R2 score on the OG Test set:"
    f"{r2_og:.3f}" #0.021
)
print("")
r2_clean = r2_score(y_clean_test, pred_clean)
print(
    "R2 score on the Clean Test set:"
    f"{r2_clean:.3f}" #0.031
)
print("")
r2_reduced = r2_score(y_reduced_test, pred_reduced)
print(
    "R2 score on the Reduced Test set:"
    f"{r2_reduced:.3f}" #0.021
)
print("")
r2_lasso = r2_score(y_reduced_test, pred_lasso)
print(
    "R2 score on the LASSO Test set:"
    f"{r2_lasso:.3f}" #0.014
)
print("")
r2_elastic = r2_score(y_reduced_test, pred_elastic)
print(
    "R2 score on the Elastic Test set:"
    f"{r2_elastic:.3f}" #0.022
)


Mean squared error on the OG Test set: 623.246

Mean squared error on the Clean Test set: 0.011

Mean squared error on the Reduced Test set: 622.856

Mean squared error on the LASSO Test set: 627.375

Mean squared error on the Elastic Test set: 622.529

R2 score on the OG Test set:0.021

R2 score on the Clean Test set:0.031

R2 score on the Reduced Test set:0.021

R2 score on the LASSO Test set:0.014

R2 score on the Elastic Test set:0.022

